In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import multiprocessing
import time
import mtkahypar


BASE = Path(
    "/Users/stevehalley/Code/Github/QuantumHypergraphPartitioning"
)

DATASET = BASE / "data" / "poisson_16_12_3"

P_DIR = DATASET / "P"
W_DIR = DATASET / "w"
OUT_DIR = DATASET / "hmetis"

OUT_DIR.mkdir(exist_ok=True)

RESULTS_CSV = DATASET / "mtkahypar_results.csv"


def load_array(path):
    """
    Load .npy, .csv, or whitespace-delimited text.
    """
    if path.suffix == ".npy":
        return np.load(path)

    elif path.suffix == ".csv":
        return np.loadtxt(path, delimiter=",")

    else:
        return np.loadtxt(path)


def find_matching_w(P_path):
    """
    Find corresponding weight file in w/ folder.
    """

    # First try exact same filename
    exact = W_DIR / P_path.name

    if exact.exists():
        return exact

    # Otherwise match by filename stem
    candidates = [
        p for p in W_DIR.iterdir()
        if p.is_file() and p.stem == P_path.stem
    ]

    if candidates:
        return candidates[0]

    return None


def convert_to_hmetis(P_path, w_path):
    """
    Convert stochastic incidence matrix P and edge weights w
    into a weighted hMetis file.

    Returns:
        hgr_path
        P
        w
        scale
        hyperedges
    """

    P = np.asarray(load_array(P_path))
    w = np.asarray(load_array(w_path)).reshape(-1)

    # rows = vertices
    # cols = hyperedges
    n_vertices, n_hyperedges = P.shape

    if len(w) != n_hyperedges:
        raise ValueError(
            f"{P_path.name}: P contains {n_hyperedges} hyperedges, "
            f"but w contains {len(w)} weights."
        )

    # Extract hyperedge supports
    hyperedges = []

    for e in range(n_hyperedges):
        vertices = np.where(P[:, e] > 0)[0]
        hyperedges.append(vertices.tolist())

    # Mt-KaHyPar uses integer weights
    if np.allclose(w, np.round(w)):
        scale = 1
        w_int = np.round(w).astype(int)

    else:
        scale = 1_000_000
        w_int = np.round(w * scale).astype(int)

    # Write .hgr
    hgr_path = OUT_DIR / f"{P_path.stem}.hgr"

    with open(hgr_path, "w") as f:

        # 01 = hyperedge weights present,
        #      vertex weights absent
        f.write(f"{n_hyperedges} {n_vertices} 01\n")

        for e in range(n_hyperedges):

            # hMetis uses 1-based indexing
            vertices_hmetis = [
                v + 1 for v in hyperedges[e]
            ]

            line = (
                [str(w_int[e])]
                + [str(v) for v in vertices_hmetis]
            )

            f.write(" ".join(line) + "\n")

    return (
        hgr_path,
        P,
        w,
        scale,
        hyperedges,
    )


# --------------------------------------------------
# Initialize Mt-KaHyPar once
# --------------------------------------------------

mtk = mtkahypar.initialize(
    multiprocessing.cpu_count()
)


# --------------------------------------------------
# Get all P files
# --------------------------------------------------

P_files = sorted(
    [p for p in P_DIR.iterdir() if p.is_file()]
)

if not P_files:
    raise FileNotFoundError(
        f"No files found in {P_DIR}"
    )


results = []


# --------------------------------------------------
# Process every instance
# --------------------------------------------------

for idx, P_path in enumerate(P_files):

    print("\n" + "=" * 60)
    print(
        f"Instance {idx + 1}/{len(P_files)}: "
        f"{P_path.name}"
    )
    print("=" * 60)

    w_path = find_matching_w(P_path)

    if w_path is None:
        print(
            f"Skipping {P_path.name}: "
            "no matching w file found."
        )
        continue

    try:

        # ------------------------------------------
        # Convert instance
        # ------------------------------------------

        (
            hgr_path,
            P,
            w,
            scale,
            hyperedges,
        ) = convert_to_hmetis(
            P_path,
            w_path
        )

        n_vertices, n_hyperedges = P.shape

        print(
            f"Vertices={n_vertices}, "
            f"Hyperedges={n_hyperedges}"
        )

        # ------------------------------------------
        # Create context
        # ------------------------------------------

        context = mtk.context_from_preset(
            mtkahypar.PresetType.HIGHEST_QUALITY
        )

        context.set_partitioning_parameters(
            2,                         # two blocks
            0.0,                       # exact balance
            mtkahypar.Objective.CUT    # min weighted cut
        )

        mtkahypar.set_seed(42)

        context.logging = False

        # ------------------------------------------
        # Load hypergraph
        # ------------------------------------------

        hypergraph = mtk.hypergraph_from_file(
            str(hgr_path),
            context,
            mtkahypar.FileFormat.HMETIS
        )

        # ------------------------------------------
        # Solve + time
        # ------------------------------------------

        start = time.perf_counter()

        partitioned_hg = hypergraph.partition(
            context
        )

        runtime = time.perf_counter() - start

        # ------------------------------------------
        # Extract metrics
        # ------------------------------------------

        cut_scaled = partitioned_hg.cut()

        cut_original = cut_scaled / scale

        imbalance = partitioned_hg.imbalance(
            context
        )

        block_weights = [
            partitioned_hg.block_weight(block)
            for block in partitioned_hg.blocks()
        ]

        # ------------------------------------------
        # Extract partition
        # ------------------------------------------

        partition = [
            partitioned_hg.block_id(v)
            for v in range(n_vertices)
        ]

        print(
            f"Cut={cut_original:.6f}, "
            f"Runtime={runtime:.6f}s, "
            f"Imbalance={imbalance}"
        )

        print(
            f"Partition={partition}"
        )

        # ------------------------------------------
        # Store results
        # ------------------------------------------

        results.append({
            "instance": P_path.stem,
            "P_file": P_path.name,
            "w_file": w_path.name,

            "n_vertices": n_vertices,
            "n_hyperedges": n_hyperedges,

            "mtkahypar_cut": cut_original,
            "mtkahypar_cut_scaled": cut_scaled,

            "weight_scale": scale,

            "runtime_seconds": runtime,
            "imbalance": imbalance,

            "block_0_weight": (
                block_weights[0]
                if len(block_weights) > 0
                else None
            ),

            "block_1_weight": (
                block_weights[1]
                if len(block_weights) > 1
                else None
            ),

            "partition": str(partition)
        })

    except Exception as e:

        print(
            f"ERROR processing {P_path.name}: {e}"
        )

        results.append({
            "instance": P_path.stem,
            "P_file": P_path.name,
            "w_file": (
                w_path.name
                if w_path is not None
                else None
            ),
            "error": str(e)
        })


# --------------------------------------------------
# Save results
# --------------------------------------------------

df = pd.DataFrame(results)

df.to_csv(
    RESULTS_CSV,
    index=False
)

print("\n" + "=" * 60)
print("Finished")
print("=" * 60)

print(
    f"Processed {len(df)} instances."
)

print(
    f"Results written to:\n{RESULTS_CSV}"
)

print("\nResults:")
print(df)


Instance 1/20: 01.npy
Vertices=12, Hyperedges=15
Cut=6.000000, Runtime=0.000976s, Imbalance=0.0
Partition=[1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1]

Instance 2/20: 02.npy
Vertices=12, Hyperedges=16
Cut=6.000000, Runtime=0.000747s, Imbalance=0.0
Partition=[0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1]

Instance 3/20: 03.npy
Vertices=12, Hyperedges=16
Cut=8.000000, Runtime=0.000861s, Imbalance=0.0
Partition=[1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1]

Instance 4/20: 04.npy
Vertices=12, Hyperedges=16
Cut=10.000000, Runtime=0.000730s, Imbalance=0.0
Partition=[0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0]

Instance 5/20: 05.npy
Vertices=12, Hyperedges=16
Cut=7.000000, Runtime=0.000551s, Imbalance=0.0
Partition=[0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1]

Instance 6/20: 06.npy
Vertices=12, Hyperedges=16
Cut=10.000000, Runtime=0.000792s, Imbalance=0.0
Partition=[0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0]

Instance 7/20: 07.npy
Vertices=12, Hyperedges=16
Cut=8.000000, Runtime=0.000574s, Imbalance=0.0
Partition=[1, 0, 1, 0, 0, 1, 0, 0, 1,

[WARNING] Mt-KaHyPar is already initialized
